# OpenVLA в симуляторе LIBERO: замкнутый цикл + физика, которую модель не видит

<a href="https://colab.research.google.com/github/bespatim/vla-manipulation-research/blob/experiments/libero-colab/notebooks/libero_openvla_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Что делает ноутбук:
1. Ставит симулятор **LIBERO** и библиотеки нужных версий.
2. Загружает **OpenVLA, дообученную авторами** на наборе задач LIBERO (Приложение E статьи OpenVLA).
3. Запускает модель в **замкнутом цикле**: кадр → модель → действие → физика → новый кадр.
4. На каждом шаге записывает то, чего модель **не получает**: силу контакта пальцев с предметами,
   положение предметов, суставы, камеру на запястье.
5. Сравнивает процент успеха с результатом авторов (Таблица 12: Spatial 84.7%, Object 88.4%, Goal 79.2%, Long 53.7%).

**Перед запуском:** меню *Среда выполнения → Сменить среду выполнения → **T4 GPU*** → Сохранить.
Затем запускайте ячейки по порядку (Shift+Enter).

| Шаг | Время |
|---|---|
| Установка | ~5 мин |
| Загрузка модели (~15 ГБ) | ~5–10 мин |
| Один эпизод | ~1–4 мин |

## 1. Проверка видеокарты

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print("Python", sys.version)
assert sys.version_info[:2] <= (3, 12), "tokenizers==0.19.1 нет для Python 3.13+ — нужен другой рантайм (например, Kaggle)"

## 2. Установка (~5 мин)

Версии `transformers`, `tokenizers`, `timm` закреплены — код модели OpenVLA проверяет их при загрузке.

In [ ]:
BRANCH = "experiments/libero-colab"   # после слияния в main поменять на "main"

%cd /content
!test -d vla-manipulation-research || git clone -q -b {BRANCH} https://github.com/bespatim/vla-manipulation-research.git
!test -d LIBERO || git clone -q --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
# Новые версии PyTorch по умолчанию не загружают numpy-массивы через torch.load — разрешаем явно
!sed -i 's/init_states = torch.load(init_states_path)/init_states = torch.load(init_states_path, weights_only=False)/' LIBERO/libero/libero/benchmark/__init__.py

!pip install -q "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10" "accelerate==0.30.1" bitsandbytes
!pip install -q "robosuite==1.4.1" "mujoco==3.2.7" easydict gym cloudpickle future "imageio[ffmpeg]"
!pip install -q --no-deps "bddl==1.0.1"

## 3. Симулятор без модели

Проверяем, что LIBERO рисует картинку на видеокарте (режим EGL — без монитора).

In [ ]:
import os, sys

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')

sys.path.insert(0, "/content/vla-manipulation-research/scripts")
import libero_openvla_eval as E

E.setup_libero("/content/LIBERO")

SUITE = "libero_spatial"   # libero_spatial | libero_object | libero_goal | libero_10

import matplotlib.pyplot as plt
env, instruction, init_states, n_tasks = E.make_env(SUITE, 0)
env.reset()
obs = env.set_init_state(init_states[0])
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(obs["agentview_image"][::-1, ::-1]); ax[0].set_title("внешняя камера (сырой кадр)")
ax[1].imshow(E.center_crop(E.preprocess_image(obs["agentview_image"]))); ax[1].set_title("что получает модель (224×224)")
ax[2].imshow(obs["robot0_eye_in_hand_image"][::-1, ::-1]); ax[2].set_title("камера на запястье (модель не видит)")
for a in ax: a.axis("off")
plt.suptitle(f"{SUITE}, задача 0: «{instruction}»\nзадач в наборе: {n_tasks} | целевые предметы: {env.obj_of_interest}", fontsize=10)
plt.show()
env.close()

## 4. Загрузка OpenVLA (~5–10 мин)

На T4 (15 ГБ) модель грузится в **4-битном** виде (~7 ГБ). По данным авторов, 4 бита не снижают качество
(статья, Таблица 11). На A100 автоматически включится полная точность bf16, как у авторов.

In [ ]:
policy = E.OpenVLAPolicy(SUITE)

## 5. Первый эпизод

Видео: **слева** — что видит модель, **справа** — камера на запястье. Скорость видео = реальное время
симуляции (20 шагов/с).

In [ ]:
from IPython.display import Video, display

OUT = f"/content/results/{SUITE}"
r = E.evaluate(policy, SUITE, task_ids=[0], trials_per_task=1, out_dir=OUT + "/first")
display(Video(f"{OUT}/first/{SUITE}_task00_trial00.mp4", embed=True, width=768))

## 6. Серия эпизодов и сравнение с авторами

`TASKS` — номера задач (0–9), `TRIALS` — попыток на задачу. 10 задач × 3 попытки = 30 эпизодов ≈ 1–1,5 часа.
Результаты пишутся по ходу, так что при обрыве сессии сделанное не пропадёт (если сохранить на Диск, шаг 8).

In [ ]:
TASKS = list(range(10))
TRIALS = 3

results = E.evaluate(policy, SUITE, task_ids=TASKS, trials_per_task=TRIALS, out_dir=OUT + "/series")

In [ ]:
import numpy as np
import pandas as pd

def wilson(k, n, z=1.96):
    # 95% доверительный интервал для доли успехов (Уилсон)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * (c - h), 100 * (c + h)

df = pd.read_csv(f"{OUT}/series/summary.csv")
k, n = int(df.success.sum()), len(df)
lo, hi = wilson(k, n)
print(f"Мой результат: {100 * k / n:.1f}% ({k}/{n}), 95% ДИ [{lo:.0f}%, {hi:.0f}%]")
print(f"У авторов:     {E.PAPER_SUCCESS[SUITE]}% (1500 попыток, bf16, A100)")
display(df.groupby("instruction").success.agg(["mean", "count"]).rename(columns={"mean": "успех", "count": "попыток"}))

## 7. Что происходило внутри эпизода

Для одного эпизода: высота захвата и целевого предмета, раскрытие пальцев и **сила контакта пальцев
с предметом** — сигнал, которого у модели нет.

In [ ]:
import glob

def plot_episode(csv_path):
    log = pd.read_csv(csv_path)
    objs = [c[len("force_"):] for c in log.columns if c.startswith("force_") and c != "force_other"]
    fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(log.step, log.eef_z, label="захват")
    for o in objs:
        if f"{o}_z" in log:
            ax[0].plot(log.step, log[f"{o}_z"], label=o)
    ax[0].set_ylabel("высота, м"); ax[0].legend(fontsize=8)
    ax[1].plot(log.step, log.gripper_q0 - log.gripper_q1, label="раскрытие пальцев")
    ax[1].plot(log.step, (log.act_env_6 > 0) * 0.08, "--", label="команда: закрыть")
    ax[1].set_ylabel("м"); ax[1].legend(fontsize=8)
    for o in objs:
        ax[2].plot(log.step, log[f"force_{o}"], label=o)
    ax[2].plot(log.step, log.force_other, ":", label="прочее (стол и т.п.)")
    ax[2].set_ylabel("сила контакта, Н"); ax[2].set_xlabel("шаг (20 Гц)"); ax[2].legend(fontsize=8)
    fig.suptitle(csv_path.split("/")[-1])
    plt.tight_layout(); plt.show()

episodes = sorted(glob.glob(f"{OUT}/series/*.csv"))
episodes = [e for e in episodes if not e.endswith("summary.csv")]
failed = [f"{OUT}/series/{SUITE}_task{t:02d}_trial{i:02d}.csv" for t, i in df.loc[~df.success, ["task_id", "trial"]].values]
print("неудачные эпизоды:", [f.split("/")[-1] for f in failed])
plot_episode(failed[0] if failed else episodes[0])

## 8. Сохранение результатов

Сессия Colab стирается при закрытии. Сохраните результаты на Google Диск или скачайте архив
и положите в репозиторий в `results/`.

In [ ]:
# На Google Диск:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/vla-results && cp -r /content/results/* /content/drive/MyDrive/vla-results/

In [ ]:
# Или архивом на компьютер:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download("/content/results.zip")